# Bagging

This notebook implements a **Bagging (Bootstrap Aggregating)** approach to fake news detection, following the optimization problem defined in the project specification. The objective minimises a weighted, category-balanced cross-entropy loss with regularisation.

# Optimization Problem

MIN Z

$$
\min_{\theta_{1}, \dots, \theta_{M}} \quad Z^{(e)} = - \frac{1}{N} \sum_{i=1}^{N} \alpha_{c_i} [w_1 y_i \log(F^{(e)}(x)) + w_0 (1 - y_i) \log(1 - F^{(e)}(x))] + \sum_{m=1}^{M} \lambda_{m} \Omega(\theta_{m})
$$

Where

- $e \in \{bag, boost, stack\}$
- $F(x) \in \{0, 1\}$

<br>SUBJECT TO <br><br>

$C_{1}^{bag}$: Ensemble Prediction Function

$$F(x) = \frac{1}{M} \displaystyle\sum_{m=1}^{M} f_m(x)$$

Where

- $f_m$ is trained on a bootstrap sample $B_m \subset \mathcal{D}$ with replacement

$C_2$: Feature Mapping

$$x_i = \phi (title_i, text_i, category_i, dataset_i)$$

$C_3$: Label Constraint

$$y_i \in \{0, 1\}, \qquad 0 = fake, \quad 1 = real$$

$C_4$: Category Weight

$$\alpha_{c_i} = \frac{N}{K \cdot N_{c_i}}$$

Where

- $K$ is the number of distinct categories
- $N_{c_i}$ is the number of samples in category $c_i$
- $N$ is the total number of samples

$C_5$: Class Weight

$$w_1 = \frac{N}{2N_1}, \qquad w_0 = \frac{N}{2N_0}$$

Where

- $N_1$: number of real samples
- $N_0$: number of fake samples
- $N = N_0 + N_1$


# Environment Configuration

The following code cell contains the cudf pandas magic command. This command is kept seperate so cudf.pandas does not try to reload when adding new imports.

In [1]:
%load_ext cudf.pandas

The following code cell contains the dependencies that will be used in this notebook.

In [19]:
import re
import warnings
import numpy as np
import pandas as pd
import cupy as cp

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import hstack, csr_matrix

warnings.filterwarnings('ignore')
np.random.seed(42)
cp.random.seed(42)

# Colab Configuration

Run the code cell below to download the data ingestion script from Github.

In [3]:
!wget https://raw.githubusercontent.com/3608Team10/COMP3608PROJECT/refs/heads/main/ingest_data.py

--2026-05-08 02:39:18--  https://raw.githubusercontent.com/3608Team10/COMP3608PROJECT/refs/heads/main/ingest_data.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8393 (8.2K) [text/plain]
Saving to: ‘ingest_data.py.1’

ingest_data.py.1    100%[===================>]   8.20K  --.-KB/s    in 0s      

2026-05-08 02:39:18 (45.4 MB/s) - ‘ingest_data.py.1’ saved [8393/8393]



# Data Ingestion

Load the unified fake-news DataFrame using the shared `ingest_data` script. This combines three Kaggle datasets (`bhavikjikadara`, `mahdimashayekhi`, `shawkyelgendy`) and returns a DataFrame with columns: `title`, `text`, `label`, `category`, `dataset`. Basic preprocessing (null-text removal, deduplication, category normalisation) is applied inside the loader.

In [4]:
from ingest_data import load_datasets
df = load_datasets()

------------------------------------------------------------
Fake News Dataset Ingestion
------------------------------------------------------------

Loading bhavikjikadara ...


[100593][02:39:21:342577][warning] Auto detection of compression type is supported only for file type buffers. For other buffer types, AUTO compression type assumes uncompressed input.


[bhavik] Loaded 'true.csv': 21417 rows


[100593][02:39:23:750716][warning] Auto detection of compression type is supported only for file type buffers. For other buffer types, AUTO compression type assumes uncompressed input.


[bhavik] Loaded 'fake.csv': 23481 rows

Loading mahdimashayekhi ...


...

[mahdi] Loaded 'fake_news_dataset.csv': 20000 rows

Loading shawkyelgendy ...


[100593][02:39:25:319586][warning] Auto detection of compression type is supported only for file type buffers. For other buffer types, AUTO compression type assumes uncompressed input.


[shawky] Loaded 'real.csv': 21871 rows


[100593][02:39:25:585378][warning] Auto detection of compression type is supported only for file type buffers. For other buffer types, AUTO compression type assumes uncompressed input.


[shawky] Loaded 'fake.csv': 20072 rows

Dropped 649 rows with empty/null text.
Dropped 6,650 duplicate rows.

------------------------------------------------------------
Fake News Dataset Summary
------------------------------------------------------------
Total rows: 99,542
Fake (0): 47,161
Real (1): 52,381

Rows per source:
shawkyelgendy          40,898
bhavikjikadara         38,644
mahdimashayekhi        20,000

Categories:
  Sports                 43,765
  Politics               21,635
  News                   19,811
  Health                 2,922
  Entertainment          2,889
  Technology             2,882
  Business               2,849
  Science                2,789
------------------------------------------------------------


# Text Preprocessing

Raw text from multiple sources contains noise: HTML entities, URLs, Twitter handles, punctuation and mixed casing. The `clean_text` function normalises each entry before vectorization.

The pipeline performs three steps:

1. **Clean** — lowercase, strip URLs, HTML tags, Twitter handles, and non-alpha characters
2. **Concatenate** — `title` and `text` are joined into a single `combined_text` column, preserving headline signal without polluting the body text distribution
3. **Remove stopwords** — common English stopwords (e.g. *the*, *is*, *at*) are filtered out so TF-IDF scores reflect meaningful, discriminative terms rather than grammatical filler

In [8]:
STOP_WORDS = ENGLISH_STOP_WORDS

def clean_text(s: str) -> str:
    """Normalise a raw text string for TF-IDF vectorization."""
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)
    s = re.sub(r'@\w+', ' ', s)
    s = re.sub(r'<[^>]+>', ' ', s)
    s = re.sub(r'&[a-z]+;', ' ', s)
    s = re.sub(r'[^a-z\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    tokens = [tok for tok in s.split() if tok not in STOP_WORDS]
    return ' '.join(tokens)

# cudf.pandas accelerates these apply calls on GPU where UDF is supported
df['clean_title'] = df['title'].apply(clean_text)
df['clean_text']  = df['text'].apply(clean_text)

df['combined_text'] = df['clean_title'] + ' ' + df['clean_text']

print(f"Rows after preprocessing : {len(df):,}")
print(f"\nSample combined_text:")
print(df['combined_text'].iloc[0][:300])

Rows after preprocessing : 99,542

Sample combined_text:
budget fight looms republicans flip fiscal script washington reuters head conservative republican faction congress voted month huge expansion national debt pay tax cuts called fiscal conservative sunday urged budget restraint keeping sharp pivot way republicans representative mark meadows speaking c


# Feature Engineering

Constraint $C_2$ defines the feature mapping:

$$x_i = \phi(title_i,\ text_i,\ category_i,\ dataset_i)$$

The full feature vector is constructed in three sub-steps:
- **TF-IDF** — fit a TF-IDF vectoriser on `combined_text` (unigrams + bigrams, capped at 50,000 features, sublinear TF scaling). Note: the vectoriser is fitted on the full dataframe here for vocabulary discovery; it will be re-fitted on the training split only after BAG-3 to prevent data leakage.
- **Encode category** — one-hot encode the `category` column into sparse binary columns
- **Encode dataset** — one-hot encode the `dataset` column into sparse binary columns

All three sparse matrices are horizontally stacked into the final feature matrix `X`.

## TF-IDF Vectorization

In [10]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2), 
    sublinear_tf=True,
    min_df=5,
    strip_accents='unicode'
)
X_tfidf = tfidf.fit_transform(df['combined_text'])  # Learns vocab and converts to TF-IDF sparse matrix

print(f"TF-IDF matrix shape: {X_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_):,}")

TF-IDF matrix shape: (99542, 50000)
Vocabulary size: 50,000


## Encode Categorical Features

In [14]:
cat_dummies  = pd.get_dummies(df['category'], prefix='cat').astype(float)
data_dummies = pd.get_dummies(df['dataset'],  prefix='ds').astype(float)

# Convert to CPU numpy arrays for scipy csr_matrix compatibility
X_cat  = csr_matrix(cat_dummies.values)
X_data = csr_matrix(data_dummies.values)

print(f"Category features : {X_cat.shape[1]}  -> {list(cat_dummies.columns)}")
print(f"Dataset features  : {X_data.shape[1]}  -> {list(data_dummies.columns)}")

X = hstack([X_tfidf, X_cat, X_data])
y = df['label'].values   # cuDF Series -> numpy via cudf.pandas

print(f"\nFinal feature matrix shape : {X.shape}")
print(f"Label distribution — Fake (0): {(y==0).sum():,}  Real (1): {(y==1).sum():,}")

Category features : 8  -> ['cat_Business', 'cat_Entertainment', 'cat_Health', 'cat_News', 'cat_Politics', 'cat_Science', 'cat_Sports', 'cat_Technology']
Dataset features  : 3  -> ['ds_bhavikjikadara', 'ds_mahdimashayekhi', 'ds_shawkyelgendy']

Final feature matrix shape : (99542, 50011)
Label distribution — Fake (0): 47,161  Real (1): 52,381


# Train/Test Split

The dataset is split 80/20 into training and test sets. `stratify=y` ensures the label distribution (fake/real ratio) is preserved in both splits — critical given the mild class imbalance in the dataset.

An index array is passed through the split so that `sample_weights` (computed in BAG-4 and BAG-5) can be aligned with `X_train` after the split. The test set is held out entirely and is never touched during training or tuning.

In [16]:
N = len(y)
indices = np.arange(N)  # Index array to align sample weights after split

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, indices,
    test_size=0.20,
    random_state=42,  # Fixed seed for reproducibility
    stratify=y        # Preserves fake/real ratio in both splits
)

# Keep a slice of the original dataframe for the test set (used in evaluation)
df_test = df.iloc[idx_test].copy()

print(f"Train : {X_train.shape[0]:,} samples  |  Test : {X_test.shape[0]:,} samples")
print(f"Train label ratio — Fake: {(y_train==0).mean():.3f}  Real: {(y_train==1).mean():.3f}")
print(f"Test  label ratio — Fake: {(y_test==0).mean():.3f}  Real: {(y_test==1).mean():.3f}")

Train : 79,633 samples  |  Test : 19,909 samples
Train label ratio — Fake: 0.474  Real: 0.526
Test  label ratio — Fake: 0.474  Real: 0.526


## Dimensionality Reduction

The raw TF-IDF + categorical feature matrix is very high-dimensional (50,000+ sparse features). `TruncatedSVD` (LSA — Latent Semantic Analysis) projects it down to 300 dense components, capturing the dominant variance while making the matrix compatible with cuML which requires dense float32 input.

This step must happen **after** the train/test split (fitted on training data only, then applied to test) to prevent data leakage.

In [20]:
N_COMPONENTS = 300

svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)
X_train = svd.fit_transform(X_train).astype(np.float32)   # fit on train only — no leakage
X_test  = svd.transform(X_test).astype(np.float32)        # apply same projection to test

print(f'Variance explained by {N_COMPONENTS} components: {svd.explained_variance_ratio_.sum():.3f}')
print(f'X_train shape after SVD: {X_train.shape}')
print(f'X_test  shape after SVD: {X_test.shape}')

Variance explained by 300 components: 0.675
X_train shape after SVD: (79633, 300)
X_test  shape after SVD: (19909, 300)


Transfer SVD-reduced arrays to GPU memory for cuML

In [21]:
X_train_dense = cp.asarray(X_train, dtype=cp.float32)
X_test_dense  = cp.asarray(X_test,  dtype=cp.float32)
y_train_gpu   = cp.asarray(y_train, dtype=cp.int32)

print(f'GPU arrays ready — X_train_dense: {X_train_dense.shape}, X_test_dense: {X_test_dense.shape}')

GPU arrays ready — X_train_dense: (79633, 300), X_test_dense: (19909, 300)


___